In [3]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(config_path="../../configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg.paths))

totalseg: /work/dlclarge2/ndirt-SegFM3D/data/totalseg
results: results
checkpoints: results/checkpoints



/tmp/ipykernel_1387690/4094501529.py:4: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="../../configs"):


In [ ]:
# create nnUNet_raw, nnUNet_preprocessed, nnUNet_results
import os
from pathlib import Path
import glob
dataset_name = "totalseg_ct"
totalseg_path = Path(cfg.paths.totalseg) 
mask_base_path = Path(cfg.paths.nako_dir) / "data"
img_name = "ct.nii.gz"
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_raw", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_preprocessed", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_results", exist_ok=True)

raw_ds_dir = os.path.join(cfg.paths.nnUNet_dir, f"nnUNet_raw/{dataset_name}")
os.makedirs(raw_ds_dir, exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTs"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTs"), exist_ok=True)

In [ ]:
import csv
import json
import shutil
import nibabel as nib
import numpy as np
from tqdm import tqdm

# discover class list from one subject (sorted for deterministic label indices)
sample_seg_dir = totalseg_path / "s0000" / "segmentations"
classes = sorted(p.stem.replace(".nii", "") for p in sample_seg_dir.glob("*.nii.gz"))
label_map = {"background": 0, **{cls: i + 1 for i, cls in enumerate(classes)}}
print(f"{len(classes)} classes found")

# read official train/test split
split_map = {}
with open(totalseg_path / "meta.csv") as f:
    for row in csv.DictReader(f, delimiter=";"):
        split_map[row["image_id"]] = row["split"]

# subjects present on disk
all_subjects = sorted(p.name for p in totalseg_path.iterdir() if p.is_dir())

def prepare_subject(subject_id, img_out_dir, lbl_out_dir):
    subj_path = totalseg_path / subject_id
    seg_dir = subj_path / "segmentations"

    # load reference image for shape/affine
    ref = nib.load(subj_path / "ct.nii.gz")
    label_vol = np.zeros(ref.shape, dtype=np.uint8)

    for cls, idx in label_map.items():
        if cls == "background":
            continue
        seg_file = seg_dir / f"{cls}.nii.gz"
        if seg_file.exists():
            mask = nib.load(seg_file).get_fdata(dtype=np.float32) > 0.5
            label_vol[mask] = idx

    # copy image
    img_dst = img_out_dir / f"{subject_id}_0000.nii.gz"
    if not img_dst.exists():
        shutil.copy2(subj_path / "ct.nii.gz", img_dst)

    # save merged label
    lbl_dst = lbl_out_dir / f"{subject_id}.nii.gz"
    if not lbl_dst.exists():
        nib.save(nib.Nifti1Image(label_vol, ref.affine, ref.header), lbl_dst)

imagesTr = Path(raw_ds_dir) / "imagesTr"
labelsTr = Path(raw_ds_dir) / "labelsTr"
imagesTs = Path(raw_ds_dir) / "imagesTs"
labelsTs = Path(raw_ds_dir) / "labelsTs"

train_subjects, test_subjects = [], []
for sid in all_subjects:
    split = split_map.get(sid, "train")
    if split in ("train", "val"):
        train_subjects.append(sid)
    else:
        test_subjects.append(sid)

print(f"Train: {len(train_subjects)}  Test: {len(test_subjects)}")

for sid in tqdm(train_subjects, desc="train"):
    prepare_subject(sid, imagesTr, labelsTr)

for sid in tqdm(test_subjects, desc="test"):
    prepare_subject(sid, imagesTs, labelsTs)

# write dataset.json (nnUNet v2 format)
dataset_json = {
    "channel_names": {"0": "CT"},
    "labels": label_map,
    "numTraining": len(train_subjects),
    "file_ending": ".nii.gz",
    "dataset_name": dataset_name,
}
with open(Path(raw_ds_dir) / "dataset.json", "w") as f:
    json.dump(dataset_json, f, indent=2)

print("Done. dataset.json written.")
